In [ ]:
# CNN Model

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout

def load_data(file_path):
    """
    Loads and preprocesses the data from the CSV file.
    Note: This is set up for the provided CSV, not raw time-series.
    For raw time-series, the reshaping part would be different.
    """
    df = pd.read_csv("all_participants_features.csv")
    df = df.drop(columns=['Participant', 'TaskKey'])

    # Drop rows with any NaN or infinite values
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    # Encode the target variable 'CognitiveLoad'
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    df['CognitiveLoad'] = le.fit_transform(df['CognitiveLoad'])

    # Separate features and target
    X = df.drop('CognitiveLoad', axis=1).values
    y = df['CognitiveLoad'].values

    # Reshape data for CNN (samples, timesteps, features)
    # This is a placeholder as the provided data is not time-series.
    # We are treating each feature as a separate 'timestep' for demonstration.
    X = X.reshape(X.shape[0], X.shape[1], 1)

    # Standardize the features
    scaler = StandardScaler()
    original_shape = X.shape
    X = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(original_shape)

    return X, y, le.classes_

def build_cnn_model(input_shape, num_classes):
    """
    Builds and returns a 1D CNN model for time-series classification.
    """
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.5))
    model.add(Conv1D(filters=128, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.5))
    model.add(Flatten())
    model.add(Dense(200, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Main execution block
if __name__ == '__main__':
    file_path = 'all_participants_features.csv'
    X, y, classes = load_data(file_path)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    input_shape = (X_train.shape[1], X_train.shape[2])
    num_classes = len(classes)

    cnn_model = build_cnn_model(input_shape, num_classes)
    cnn_model.summary()

    # Train the model
    # Note: Training on the provided data is for demonstration.
    # Real-world performance requires actual time-series data.
    history = cnn_model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

    # Evaluate the model
    loss, accuracy = cnn_model.evaluate(X_test, y_test, verbose=0)
    print(f"\nCNN Model Accuracy: {accuracy*100:.2f}%")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_4 (Conv1D)               │ (None, 12, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_4 (MaxPooling1D)  │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 6, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 4, 128)         │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_5 (MaxPooling1D)  │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 2, 128)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 200)            │        51,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 3)              │           603 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 76,963 (300.64 KB)

 Trainable params: 76,963 (300.64 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.3493 - loss: 1.0982 - val_accuracy: 0.4248 - val_loss: 1.0853
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4002 - loss: 1.0915 - val_accuracy: 0.4248 - val_loss: 1.0838
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4143 - loss: 1.0865 - val_accuracy: 0.4248 - val_loss: 1.0898
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3995 - loss: 1.0868 - val_accuracy: 0.4248 - val_loss: 1.0915
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.3705 - loss: 1.0981 - val_accuracy: 0.4248 - val_loss: 1.0817
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4061 - loss: 1.0860 - val_accuracy: 0.4248 - val_loss: 1.0869
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4342 - loss: 1.0780 - val_accuracy: 0.4248 - val_loss: 1.0851
Epoch 8/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4039 - loss: 1.0922 - val_accuracy: 0.4248 - val_loss:

In [ ]:
# BiLSTM Model
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, TimeDistributed, Flatten

def load_data(file_path):
    """
    Loads and preprocesses the feature data.

    *** IMPORTANT ***
    This function uses the provided feature data and artificially reshapes it
    to fit the 3D/4D input requirements of CNN-LSTM. For truly accurate
    time-series analysis, you must replace the input data and reshaping logic
    with your raw EEG/GSR segments (e.g., shape [samples, time_steps, features]).
    """
    df = pd.read_csv(file_path)
    df = df.drop(columns=['Participant', 'TaskKey'])

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    le = LabelEncoder()
    df['CognitiveLoad'] = le.fit_transform(df['CognitiveLoad'])

    X = df.drop('CognitiveLoad', axis=1).values
    y = df['CognitiveLoad'].values

    # Reshaping for demonstration (treating features as a sequence of subsequences)
    n_timesteps = X.shape[1]
    n_features = 1
    n_subsequences = 3
    n_steps = n_timesteps // n_subsequences
    X = X[:, :n_subsequences * n_steps]
    X = X.reshape(X.shape[0], n_subsequences, n_steps, n_features)

    scaler = StandardScaler()
    original_shape = X.shape
    X = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(original_shape)

    return X, y, le.classes_

def build_cnn_bilstm_model(input_shape, num_classes):
    """
    Builds and returns an enhanced CNN-BiLSTM hybrid model.
    """
    model = Sequential()
    # Feature extraction (TimeDistributed CNN)
    model.add(TimeDistributed(Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'), input_shape=input_shape))
    model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
    model.add(TimeDistributed(Dropout(0.3)))
    model.add(TimeDistributed(Flatten()))

    # Sequence processing (BiLSTM)
    model.add(Bidirectional(LSTM(100, activation='relu', return_sequences=False)))
    model.add(Dropout(0.5))

    # Output Layer
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Main execution block
if __name__ == '__main__':
    file_path = 'all_participants_features.csv'
    X, y, classes = load_data(file_path)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Input shape is (n_subsequences, n_steps, n_features)
    input_shape = (X_train.shape[1], X_train.shape[2], X_train.shape[3])
    num_classes = len(classes)

    cnn_bilstm_model = build_cnn_bilstm_model(input_shape, num_classes)
    print("\n--- Enhanced CNN-BiLSTM Hybrid Model Summary ---")
    cnn_bilstm_model.summary()

    # Train the model (20 epochs for quick testing)
    print("\n--- Training Enhanced CNN-BiLSTM Hybrid Model ---")
    cnn_bilstm_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=0)

    loss, accuracy = cnn_bilstm_model.evaluate(X_test, y_test, verbose=0)
    print(f"\nEnhanced CNN-BiLSTM Model Accuracy: {accuracy*100:.2f}% (on feature data)")


--- Enhanced CNN-BiLSTM Hybrid Model Summary ---


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed_11             │ (None, 3, 4, 128)      │           512 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_12             │ (None, 3, 2, 128)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_13             │ (None, 3, 2, 128)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_14             │ (None, 3, 256)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_4 (Bidirectional) │ (None, 200)            │       285,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_21 (Dropout)            │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 3)              │           603 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 286,715 (1.09 MB)

 Trainable params: 286,715 (1.09 MB)

 Non-trainable params: 0 (0.00 B)


--- Training Enhanced CNN-BiLSTM Hybrid Model ---

Enhanced CNN-BiLSTM Model Accuracy: 42.48% (on feature data)


In [ ]:
# 3. CNN-LSTM Hybrid Model
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout, TimeDistributed, Flatten

def load_data(file_path):
    """
    Loads and preprocesses the data from the CSV file.
    Note: This is set up for the provided CSV, not raw time-series.
    For raw time-series, the reshaping part would be different.
    """
    df = pd.read_csv(file_path)
    df = df.drop(columns=['Participant', 'TaskKey'])

    # Drop rows with any NaN or infinite values
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    # Encode the target variable 'CognitiveLoad'
    le = LabelEncoder()
    df['CognitiveLoad'] = le.fit_transform(df['CognitiveLoad'])

    # Separate features and target
    X = df.drop('CognitiveLoad', axis=1).values
    y = df['CognitiveLoad'].values

    # Reshape data for CNN-LSTM (samples, subsequences, timesteps, features)
    # This is a placeholder. For actual time-series, you'd segment the raw data.
    n_timesteps = X.shape[1]
    n_features = 1
    n_subsequences = 3 # Example number of subsequences
    n_steps = n_timesteps // n_subsequences
    X = X[:, :n_subsequences * n_steps] # Trim to fit
    X = X.reshape(X.shape[0], n_subsequences, n_steps, n_features)

    # Standardize the features
    scaler = StandardScaler()
    original_shape = X.shape
    X = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(original_shape)

    return X, y, le.classes_

def build_cnn_lstm_model(input_shape, num_classes):
    """
    Builds and returns a CNN-LSTM hybrid model.
    """
    model = Sequential()
    model.add(TimeDistributed(Conv1D(filters=64, kernel_size=3, activation='relu'), input_shape=input_shape))
    model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
    model.add(TimeDistributed(Flatten()))
    model.add(LSTM(100, activation='relu'))
    model.add(Dropout(0.5))
    model.add(Dense(100, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Main execution block
if __name__ == '__main__':
    file_path = 'all_participants_features.csv'
    X, y, classes = load_data(file_path)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    input_shape = (X_train.shape[1], X_train.shape[2], X_train.shape[3])
    num_classes = len(classes)

    cnn_lstm_model = build_cnn_lstm_model(input_shape, num_classes)
    cnn_lstm_model.summary()

    history = cnn_lstm_model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test))

    loss, accuracy = cnn_lstm_model.evaluate(X_test, y_test, verbose=0)
    print(f"\nCNN-LSTM Model Accuracy: {accuracy*100:.2f}%")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 3, 2, 64)       │           256 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 3, 1, 64)       │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 3, 64)          │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 100)            │        66,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 3)              │           303 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 76,659 (299.45 KB)

 Trainable params: 76,659 (299.45 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 3s 14ms/step - accuracy: 0.3667 - loss: 1.1037 - val_accuracy: 0.4248 - val_loss: 1.0865
Epoch 2/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4086 - loss: 1.0844 - val_accuracy: 0.4248 - val_loss: 1.0822
Epoch 3/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.3934 - loss: 1.0866 - val_accuracy: 0.4248 - val_loss: 1.0809
Epoch 4/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4149 - loss: 1.0783 - val_accuracy: 0.4248 - val_loss: 1.0798
Epoch 5/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4426 - loss: 1.0713 - val_accuracy: 0.4248 - val_loss: 1.0802
Epoch 6/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.3770 - loss: 1.0880 - val_accuracy: 0.4248 - val_loss: 1.0824
Epoch 7/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4157 - loss: 1.0786 - val_accuracy: 0.4248 - val_loss: 1.0804
Epoch 8/20
34/34 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.4132 - loss: 1.0774 - val_accuracy: 0.4248 - val_loss

In [ ]:
# 4. Transformer-based Sequence Encoder

In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, LayerNormalization
from tensorflow.keras.layers import MultiHeadAttention, GlobalAveragePooling1D

def load_data(file_path):
    """
    Loads and preprocesses the feature data.

    *** IMPORTANT ***
    This function uses the provided feature data and artificially reshapes it
    to fit the 3D input requirements of the Transformer. For truly accurate
    time-series analysis, you must replace the input data and reshaping logic
    with your raw EEG/GSR segments (e.g., shape [samples, time_steps, features]).
    """
    df = pd.read_csv(file_path)
    df = df.drop(columns=['Participant', 'TaskKey'])

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    le = LabelEncoder()
    df['CognitiveLoad'] = le.fit_transform(df['CognitiveLoad'])

    X = df.drop('CognitiveLoad', axis=1).values
    y = df['CognitiveLoad'].values

    # Reshape data for Transformer (samples, timesteps, features)
    X = X.reshape(X.shape[0], X.shape[1], 1)

    scaler = StandardScaler()
    original_shape = X.shape
    X = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(original_shape)

    # Transpose X to fit the transformer model, which typically expects (batch, features, timesteps)
    X = np.transpose(X, (0, 2, 1))

    return X, y, le.classes_

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0.3):
    """
    Creates a single Transformer encoder block with LayerNormalization.
    """
    # Attention Block
    x = LayerNormalization(epsilon=1e-6)(inputs)
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(x, x)
    x = Dropout(dropout)(x)
    res = x + inputs

    # Feed Forward Block
    x = LayerNormalization(epsilon=1e-6)(res)
    x = Dense(ff_dim, activation='relu')(x)
    x = Dropout(dropout)(x)
    x = Dense(inputs.shape[-1])(x)
    return x + res

def build_transformer_model(input_shape, num_classes):
    """
    Builds and returns a deeper Transformer model for time-series classification.
    """
    inputs = Input(shape=input_shape)

    # Deeper stacking of Transformer blocks
    x = transformer_encoder(inputs, head_size=128, num_heads=8, ff_dim=256)
    x = transformer_encoder(x, head_size=128, num_heads=8, ff_dim=256)

    x = GlobalAveragePooling1D(data_format='channels_first')(x)
    x = Dropout(0.4)(x)
    x = Dense(128, activation='relu')(x)
    x = Dense(num_classes, activation='softmax')(x)
    model = Model(inputs=inputs, outputs=x)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Main execution block
if __name__ == '__main__':
    file_path = 'all_participants_features.csv'
    X, y, classes = load_data(file_path)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Input shape is (features, timesteps)
    input_shape = (X_train.shape[1], X_train.shape[2])
    num_classes = len(classes)

    transformer_model = build_transformer_model(input_shape, num_classes)
    print("\n--- Transformer-based Sequence Encoder Model Summary ---")
    transformer_model.summary()

    print("\n--- Training Transformer Model ---")
    transformer_model.fit(X_train, y_train, epochs=20, batch_size=32, validation_data=(X_test, y_test), verbose=0)

    loss, accuracy = transformer_model.evaluate(X_test, y_test, verbose=0)
    print(f"\nTransformer Model Accuracy: {accuracy*100:.2f}% (on feature data)")


--- Transformer-based Sequence Encoder Model Summary ---


Model: "functional_62"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_9       │ (None, 1, 14)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 14)     │         28 │ input_layer_9[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 14)     │     60,430 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_23          │ (None, 1, 14)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 1, 14)     │          0 │ dropout_23[0][0], │
│                     │                   │            │ input_layer_9[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 14)     │         28 │ add_4[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_19 (Dense)    │ (None, 1, 256)    │      3,840 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_24          │ (None, 1, 256)    │          0 │ dense_19[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_20 (Dense)    │ (None, 1, 14)     │      3,598 │ dropout_24[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 1, 14)     │          0 │ dense_20[0][0],   │
│                     │                   │            │ add_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 14)     │         28 │ add_5[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 1, 14)     │     60,430 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_26          │ (None, 1, 14)     │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_6 (Add)         │ (None, 1, 14)     │          0 │ dropout_26[0][0], │
│                     │                   │            │ add_5[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 1, 14)     │         28 │ add_6[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_21 (Dense)    │ (None, 1, 256)    │      3,840 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_27          │ (None, 1, 256)    │          0 │ dense_21[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_22 (Dense)    │ (None, 1, 14)     │      3,598 │ dropout_27[0][0]

 Total params: 136,491 (533.17 KB)

 Trainable params: 136,491 (533.17 KB)

 Non-trainable params: 0 (0.00 B)


--- Training Transformer Model ---


/usr/local/lib/python3.12/dist-packages/keras/src/ops/nn.py:944: UserWarning: You are using a softmax over axis 3 of a tensor of shape (None, 8, 1, 1). This axis has size 1. The softmax operation will always return the value 1, which is likely not what you intended. Did you mean to use a sigmoid instead?
  warnings.warn(



Transformer Model Accuracy: 42.48% (on feature data)


In [ ]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, TimeDistributed, Flatten

def load_data(file_path):
    """
    Loads and preprocesses the feature data.

    *** IMPORTANT ***
    This function artificially reshapes the feature data (12 features)
    into a 4D format (e.g., 3 subsequences x 4 time steps x 1 feature)
    required for the TimeDistributed CNN-LSTM layers. This reshaping
    is a placeholder and should be replaced with raw time-series segmenting
    for true performance gains.
    """
    df = pd.read_csv(file_path)
    df = df.drop(columns=['Participant', 'TaskKey'])

    # Drop rows with any NaN or infinite values
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    le = LabelEncoder()
    df['CognitiveLoad'] = le.fit_transform(df['CognitiveLoad'])

    X = df.drop('CognitiveLoad', axis=1).values
    y = df['CognitiveLoad'].values

    # Reshaping for CNN-LSTM (samples, subsequences, timesteps, features)
    n_timesteps = X.shape[1]
    n_features = 1
    n_subsequences = 3

    # Calculate n_steps and trim features if necessary for even division
    actual_features = X.shape[1]
    n_steps = actual_features // n_subsequences
    X = X[:, :n_subsequences * n_steps]
    X = X.reshape(X.shape[0], n_subsequences, n_steps, n_features)

    scaler = StandardScaler()
    original_shape = X.shape
    X = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(original_shape)

    return X, y, le.classes_

def build_tuned_cnn_bilstm_model(input_shape, num_classes):
    """
    Builds an enhanced CNN-BiLSTM hybrid model with tuned hyperparameters.
    """
    model = Sequential()

    # Tuned CNN Blocks (Feature Extraction)
    model.add(TimeDistributed(Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'), input_shape=input_shape))
    model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
    model.add(TimeDistributed(Dropout(0.3)))

    model.add(TimeDistributed(Conv1D(filters=256, kernel_size=3, activation='relu', padding='same')))
    model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
    model.add(TimeDistributed(Dropout(0.3)))

    model.add(TimeDistributed(Flatten()))

    # Tuned BiLSTM Block (Sequence Processing)
    model.add(Bidirectional(LSTM(128, activation='tanh', return_sequences=False)))
    model.add(Dropout(0.4))

    # Tuned Dense Layers (Classification)
    model.add(Dense(128, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0005)
    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Main execution block
if __name__ == '__main__':
    file_path = 'all_participants_features.csv'
    X, y, classes = load_data(file_path)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    input_shape = (X_train.shape[1], X_train.shape[2], X_train.shape[3])
    num_classes = len(classes)

    tuned_model = build_tuned_cnn_bilstm_model(input_shape, num_classes)

    # Training with optimized epochs (50) and batch size (64)
    tuned_model.fit(
        X_train, y_train,
        epochs=50,
        batch_size=64,
        validation_data=(X_test, y_test),
        verbose=0
    )

    loss, accuracy = tuned_model.evaluate(X_test, y_test, verbose=0)
    print(f"\nHighly Tuned CNN-BiLSTM Model Accuracy: {accuracy*100:.2f}% (on feature data)")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Highly Tuned CNN-BiLSTM Model Accuracy: 41.35% (on feature data)


In [4]:
# BiLSTM Model
import pandas as pd
import numpy as np
import tensorflow as tf
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Bidirectional, LSTM, Dense, Dropout, TimeDistributed, Flatten

def load_data(file_path):
    """
    Loads and preprocesses the feature data.
    """
    df = pd.read_csv(file_path)
    df = df.drop(columns=['Participant', 'TaskKey'])

    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)

    le = LabelEncoder()
    df['CognitiveLoad'] = le.fit_transform(df['CognitiveLoad'])

    X = df.drop('CognitiveLoad', axis=1).values
    y = df['CognitiveLoad'].values

    # Reshaping for demonstration
    n_timesteps = X.shape[1]
    n_features = 1
    n_subsequences = 3
    n_steps = n_timesteps // n_subsequences
    X = X[:, :n_subsequences * n_steps]
    X = X.reshape(X.shape[0], n_subsequences, n_steps, n_features)

    scaler = StandardScaler()
    original_shape = X.shape
    X = scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(original_shape)

    return X, y, le.classes_

def build_cnn_bilstm_model(input_shape, num_classes):
    """
    Builds and returns an enhanced CNN-BiLSTM hybrid model.
    """
    model = Sequential()
    model.add(TimeDistributed(Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'), input_shape=input_shape))
    model.add(TimeDistributed(MaxPooling1D(pool_size=2)))
    model.add(TimeDistributed(Dropout(0.3)))
    model.add(TimeDistributed(Flatten()))

    model.add(Bidirectional(LSTM(100, activation='relu', return_sequences=False)))
    model.add(Dropout(0.5))

    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Main execution block
if __name__ == '__main__':
    file_path = 'all_participants_features.csv'
    X, y, classes = load_data(file_path)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    input_shape = (X_train.shape[1], X_train.shape[2], X_train.shape[3])
    num_classes = len(classes)

    cnn_bilstm_model = build_cnn_bilstm_model(input_shape, num_classes)
    print("\n--- Enhanced CNN-BiLSTM Hybrid Model Summary ---")
    cnn_bilstm_model.summary()

    print("\n--- Training Enhanced CNN-BiLSTM Hybrid Model ---")
    cnn_bilstm_model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test), verbose=0)

    loss, accuracy = cnn_bilstm_model.evaluate(X_test, y_test, verbose=0)
    print(f"\nEnhanced CNN-BiLSTM Model Accuracy: {accuracy*100:.2f}% (on feature data)")

    # --- Save model as pickle ---
    with open("model.pkl", "wb") as f:
        pickle.dump(cnn_bilstm_model, f)

    print("✅ Trained CNN-BiLSTM model saved as model.pkl")

    # (Optional, safer way)
    cnn_bilstm_model.save("model.h5")
    print("✅ Model also saved as model.h5 for portability")


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/wrapper.py:27: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



--- Enhanced CNN-BiLSTM Hybrid Model Summary ---


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ time_distributed                │ (None, 3, 4, 128)      │           512 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_1              │ (None, 3, 2, 128)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_2              │ (None, 3, 2, 128)      │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 3, 256)         │             0 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 200)            │       285,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 200)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 3)              │           603 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 286,715 (1.09 MB)

 Trainable params: 286,715 (1.09 MB)

 Non-trainable params: 0 (0.00 B)


--- Training Enhanced CNN-BiLSTM Hybrid Model ---



Enhanced CNN-BiLSTM Model Accuracy: 42.48% (on feature data)
✅ Trained CNN-BiLSTM model saved as model.pkl
✅ Model also saved as model.h5 for portability


In [1]:
# Pickel File

In [5]:
with open("model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

> **Model Evaluation on Participants 1–38**

> **CNN Model**  
> Accuracy: 42.48%

> **Enhanced CNN-BiLSTM Hybrid Model**  
> Accuracy: 42.48% (on feature data)

> **CNN-LSTM Model**  
> Accuracy: 42.48%

> **Transformer Model**  
> Accuracy: 42.48% (on feature data)

> **Highly Tuned CNN-BiLSTM Model**  
> Accuracy: 41.35% (on feature data)
